# Homicide Clearance Prediction

This notebook analyzes homicide records to predict whether a case was **solved (cleared)**.

**Steps:**
1. Load data
2. Clean data (remove unknown values coded as 999)
3. Encode categorical variables
4. Train a Logistic Regression model
5. Evaluate accuracy
6. Plot clearance rate over time

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

## 1. Load Dataset

In [ ]:
df = pd.read_csv('homicide_data.csv')
print(f"Shape: {df.shape}")
df.head()

## 2. Clean Data

Remove rows where any categorical column contains the code **999** (unknown).

In [ ]:
# Remove rows with unknown values (coded as 999)
df_clean = df[(df['VicSex'] != 999) & (df['VicRace'] != 999) & (df['Weapon'] != 999)]
print(f"Rows before cleaning: {len(df)}")
print(f"Rows after cleaning:  {len(df_clean)}")
df_clean.head()

## 3. Encode Categorical Variables

In [ ]:
# Map numeric codes to labels
sex_map    = {1: 'Male', 2: 'Female'}
race_map   = {1: 'White', 2: 'Black', 3: 'Hispanic', 4: 'Other'}
weapon_map = {1: 'Firearm', 2: 'Knife', 3: 'Blunt Object', 4: 'Other'}

df_clean = df_clean.copy()
df_clean['VicSex']  = df_clean['VicSex'].map(sex_map)
df_clean['VicRace'] = df_clean['VicRace'].map(race_map)
df_clean['Weapon']  = df_clean['Weapon'].map(weapon_map)

# Label-encode each column with its own encoder
for col in ['VicSex', 'VicRace', 'Weapon']:
    df_clean[col] = LabelEncoder().fit_transform(df_clean[col])

df_clean.head()

## 4. Train Logistic Regression Model

In [ ]:
features = ['Year', 'VicSex', 'VicRace', 'Weapon']
target   = 'Solved'

X = df_clean[features]
y = df_clean[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

print("Model trained successfully.")

## 5. Evaluate Model

In [ ]:
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {acc:.2%}")

## 6. Clearance Rate Over Time

In [ ]:
clearance_by_year = df_clean.groupby('Year')['Solved'].mean()

plt.figure(figsize=(10, 5))
plt.plot(clearance_by_year.index, clearance_by_year.values, marker='o', linewidth=2, color='steelblue')
plt.title('Homicide Clearance Rate by Year')
plt.xlabel('Year')
plt.ylabel('Clearance Rate')
plt.ylim(0, 1)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()